# Flight duration model: Regularization!

In the previous exercise you added more predictors to the flight duration model. The model performed well on testing data, but with so many coefficients it was difficult to interpret.

In this exercise you'll use Lasso regression (regularized with a L1 penalty) to create a more parsimonious model. Many of the coefficients in the resulting model will be set to zero. This means that only a subset of the predictors actually contribute to the model. Despite the simpler model, it still produces a good RMSE on the testing data.

You'll use a specific value for the regularization strength. Later you'll learn how to find the best value using cross validation.

The data (same as previous exercise) are available as `flights`, randomly split into `flights_train` and `flights_test`.

There are two parameters for this model, λ (`regParam`) and α (`elasticNetParam`), where α determines the type of regularization and λ gives the strength of regularization.

## Instructions

- Fit a linear regression model to the training data. Set the regularization strength to 1.
- Calculate the RMSE on the testing data.
- Look at the model coefficients.
- How many of the coefficients are equal to zero?

In [1]:
# # Import the SparkSession class
# import pyspark
# from pyspark.sql import SparkSession

# spark = SparkSession.builder.appName('flights').getOrCreate()


In [1]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [2]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [3]:
flights = spark.read.csv('file:///home/talentum/test-jupyter/c5-MLWithPySpark/M3-Regression/4_Regularization/dataset/flights-1000.csv',
                         sep=',',
						 header=True,
						 inferSchema=True,
						 nullValue='NA')

print(flights.printSchema())

root
 |-- mon: integer (nullable = true)
 |-- dom: integer (nullable = true)
 |-- dow: integer (nullable = true)
 |-- carrier: string (nullable = true)
 |-- flight: integer (nullable = true)
 |-- org: string (nullable = true)
 |-- mile: integer (nullable = true)
 |-- depart: double (nullable = true)
 |-- duration: integer (nullable = true)
 |-- delay: integer (nullable = true)

None


In [4]:
from pyspark.sql.functions import round

flights = flights.withColumn('km', round(flights.mile * 1.60934, 0)).drop('mile')

from pyspark.ml.feature import StringIndexer

flights = StringIndexer(inputCol='org', outputCol='org_idx').fit(flights).transform(flights)

from pyspark.ml.feature import Bucketizer, OneHotEncoder, OneHotEncoderEstimator

# onehot = OneHotEncoder(inputCols=['org_idx'], outputCols=['org_dummy'])
onehot = OneHotEncoderEstimator(inputCols=['org_idx'], outputCols=['org_dummy'])
flights = onehot.fit(flights).transform(flights)
buckets = Bucketizer(splits=[0, 3, 6, 9, 12, 15, 18, 21, 24], inputCol='depart',\
outputCol='depart_bucket')
flights = buckets.transform(flights)
# onehot = OneHotEncoder(inputCols=['depart_bucket'], outputCols=['depart_dummy'])
onehot = OneHotEncoderEstimator(inputCols=['depart_bucket'], outputCols=['depart_dummy'])
flights = onehot.fit(flights).transform(flights)
# onehot = OneHotEncoder(inputCols=['dow', 'mon'], outputCols=['dow_dummy', 'mon_dummy'])
onehot = OneHotEncoderEstimator(inputCols=['dow', 'mon'], outputCols=['dow_dummy', 'mon_dummy'])
flights = onehot.fit(flights).transform(flights)

from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=['km', 'org_dummy', 'depart_dummy', 'dow_dummy', 'mon_dummy']\
                            , outputCol='features')

flights = assembler.transform(flights)
flights_train, flights_test = flights.randomSplit([0.8, 0.2], seed=13)

In [ ]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Fit Lasso model (λ = 1, α = 1) to training data
regression = ____(____, ____, elasticNetParam=1).____(____)

# Calculate the RMSE on testing data
rmse = ____(____).____(____)
print("The test RMSE is", rmse)

# Look at the model coefficients
coeffs = regression.____
print(coeffs)

# Number of zero coefficients
zero_coeff = sum([____ for beta in regression.coefficients])
print("Number of coefficients equal to 0:", zero_coeff)

In [5]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Fit Lasso model (λ = 1, α = 1) to training data
regression = LinearRegression(labelCol = 'duration', regParam = 1, elasticNetParam=1)\
.fit(flights_train)

# Calculate the RMSE on testing data
rmse = RegressionEvaluator(labelCol = 'duration')\
.evaluate(regression.transform(flights_test))
print("The test RMSE is", rmse)

# Look at the model coefficients
coeffs = regression.coefficients
print(coeffs)

# Number of zero coefficients
zero_coeff = sum([beta == 0 for beta in regression.coefficients])
print("Number of coefficients equal to 0:", zero_coeff)

The test RMSE is 10.984343771570359
[0.07389402740392768,2.291440243753814,-0.8760381259475464,23.86049085966021,17.50514370588225,-6.393301619531375,-1.491688863088528,-18.580714413668534,0.0,0.0,-0.2458206429833692,0.0,0.0,0.14127986333000958,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.1202706248270329,0.0,0.0,0.0,0.0,0.0,0.0]
Number of coefficients equal to 0: 21


Regularization produced a far simpler model with similar test performance.